# PHASE-13 — REGIONAL INTEGRATION GATE REVIEW (PART 1E-B)

**READ-ONLY.** No inference, no model construction, no CUDA, no proposal generation, no oracle rerun,
no TEST access. This notebook only reads the frozen gate artifact and its bound evidence.

**Gate id:** `REGIONAL_INTEGRATION_GATE_AFTER_PHASE13`
**Adjudication mode:** `QUALITATIVE_PROSPECTIVE_EVIDENCE_ADJUDICATION`
**Evidence scope:** prospectively frozen evidence only (criteria frozen in Part 1E-A **before** any
Part-1D-B result existed).

> This notebook does not duplicate the hundreds of rows already visible in the Part-1D-B reviewer
> notebook. It presents the *adjudication record*.

In [1]:
import json
import hashlib
from pathlib import Path

import pandas as pd

ROOT = Path(r'D:\Softfors\jup')
AUDITS = ROOT / 'reports' / 'audits'
TABLES = ROOT / 'reports' / 'tables'


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''):
            h.update(c)
    return h.hexdigest()


gate = json.loads((AUDITS / 'phase13_regional_integration_gate_after_phase13.json').read_text('utf-8'))
crit = json.loads((AUDITS / 'phase13_regional_integration_gate_criteria_frozen.json').read_text('utf-8'))
ev = json.loads((AUDITS / 'phase13_part1d_b_predicted_region_evaluation.json').read_text('utf-8'))
dim = pd.read_csv(TABLES / 'phase13_regional_integration_gate_after_phase13.csv')
print('loaded frozen gate artifact + bound evidence')

loaded frozen gate artifact + bound evidence


## 1. Prospective criteria identity

The criteria were frozen in Part 1E-A before any Part-1D-B result existed, and were **not edited or
reinterpreted** during adjudication.

In [2]:
print('status                          :', crit['status'])
print('part1d_b_results_inspected      :', crit['part1d_b_results_inspected'])
print('numerical_thresholds_introduced :', crit['numerical_thresholds_introduced'])
print('gate mode                       :', crit['gate_mode'])
print('dimensions                      :', list(crit['dimensions']))
print('outcomes                        :', list(crit['outcomes']))
print('criteria SHA-256                :', sha256_file(AUDITS / 'phase13_regional_integration_gate_criteria_frozen.json'))
print('recorded in gate artifact       :', gate['bindings']['prospective_criteria']['sha'])
print('criteria edited by Part 1E-B    :', gate['bindings']['prospective_criteria']['edited'])

status                          : PROSPECTIVE_GATE_CRITERIA_FROZEN_BEFORE_PART1D_B_RESULTS
part1d_b_results_inspected      : NO
numerical_thresholds_introduced : 0
gate mode                       : QUALITATIVE_PROSPECTIVE_EVIDENCE_ADJUDICATION
dimensions                      : ['A', 'B', 'C', 'D']
outcomes                        : ['GO', 'REVISE_LOCALIZER', 'NO_GO', 'CHECK_REQUIRED']
criteria SHA-256                : 342a8bcac2c87bf0b85f7bbd963179e74f029825c0434cf527fbd939d2d13c21
recorded in gate artifact       : 342a8bcac2c87bf0b85f7bbd963179e74f029825c0434cf527fbd939d2d13c21
criteria edited by Part 1E-B    : NO


## 2. Integrity precondition

A material integrity failure would have forced `CHECK_REQUIRED` with **no** verdict issued.

In [3]:
pre = gate['integrity_precondition']
print('integrity precondition result :', pre['result'])
print()
for k, v in pre['checks'].items():
    print(f'  {"PASS" if v else "FAIL"}  {k}')

integrity precondition result : PASS

  PASS  gate_criteria_sha_match
  PASS  gate_status
  PASS  gate_part1d_b_results_inspected_no
  PASS  gate_numerical_thresholds_zero
  PASS  gate_mode
  PASS  part1d_b_status_pass
  PASS  global_reproduction_all_four_pass
  PASS  case_model_rows_412
  PASS  component_model_rows_608
  PASS  model_summary_rows_4
  PASS  test_reads_zero
  PASS  integrity_failures_empty
  PASS  r1_protocol_sha_match
  PASS  part1d_b_integrity_checks_all_pass


## 3. Dimension A — `ORACLE_MECHANISM_VALUE`

Part-1D-B may change A **only** if it reveals a material integrity / reproduction defect in Phase-12.
Poor predicted-localizer performance must **not** be used to invalidate A.

In [4]:
A = gate['dimensions']['A']
print('pre-state    :', A['pre_state'])
print('state        :', A['state'])
print('changed      :', 'NO')
print()
print('frozen Phase-12 oracle Dice gains (re-confirmed):')
for k, v in A['oracle_gains'].items():
    if k in ('unet', 'attention_unet', 'unetpp', 'deeplabv3plus'):
        print(f'  {k:<16} {v:+.6f}')
print()
print(A['evidence'])

pre-state    : SUPPORTED_DESCRIPTIVE_EVIDENCE
state        : SUPPORTED_DESCRIPTIVE_EVIDENCE
changed      : NO

frozen Phase-12 oracle Dice gains (re-confirmed):
  unet             +0.130542
  attention_unet   +0.069790
  unetpp           +0.086164
  deeplabv3plus    +0.190121

Phase-12 oracle Dice gains remain positive for all four architectures and were independently re-confirmed through the frozen oracle gains: UNet +0.130542, Attention U-Net +0.069790, UNet++ +0.086164, DeepLabV3+ +0.190121. Part-1D-B global reproduction reproduced the shared global path at machine precision (max abs per-case Dice difference 5.551e-17 to 1.110e-16), so the comparator provenance is intact. No material Phase-12 integrity or reproduction defect was discovered.


## 4. Dimension B — `PRACTICAL_LOCALIZATION_RETENTION`

Adjudicated from multiple frozen fields; **no single field alone decides** and **no retention cutoff
was introduced**. Negative retention values are preserved unclipped.

In [5]:
B = gate['dimensions']['B']
print('pre-state :', B['pre_state'])
print('state     :', B['state'])
print()
rows = []
for m, b in B['per_model'].items():
    rows.append({'model': m, 'GLOBAL': b['global_dice'], 'ORACLE': b['oracle_dice'],
                 'PREDICTED': b['predicted_dice'], 'DELTA_ORACLE': b['DELTA_ORACLE_DICE'],
                 'DELTA_PRED': b['DELTA_PRED_DICE'], 'RETENTION_DICE': b['RETENTION_DICE'],
                 'ORACLE_GAP': b['ORACLE_GAP_DICE'], 'IoU_ret': b['iou_gain_retention'],
                 'ObjRecall_ret': b['object_recall_gain_retention'],
                 'FNred_ret': b['fn_reduction_retention'],
                 'recovered': b['component_recoveries'], 'lost': b['component_losses'],
                 'empty_rec': b['empty_recoveries'], 'new_empty': b['new_empty_predictions']})
print(pd.DataFrame(rows).to_string(index=False))
print()
print(B['evidence'])

pre-state : NOT_EVALUABLE_UNTIL_PHASE13
state     : WEAK__FAILS_TO_TRANSLATE_ORACLE_MECHANISM_INTO_DEPLOYABLE_UTILITY

         model   GLOBAL   ORACLE  PREDICTED  DELTA_ORACLE  DELTA_PRED  RETENTION_DICE  ORACLE_GAP   IoU_ret  ObjRecall_ret  FNred_ret  recovered  lost  empty_rec  new_empty
          unet 0.594180 0.724722   0.302314      0.130542   -0.291866       -2.235800   -0.422408 -2.503471       0.488372   0.041765         16     0          8          0
attention_unet 0.629805 0.699595   0.308619      0.069790   -0.321186       -4.602209   -0.390976 -5.373957       0.310345  -0.096985          9     0          3          0
        unetpp 0.640516 0.726679   0.311355      0.086164   -0.329161       -3.820171   -0.415324 -4.796426       0.500000   0.090420         14     0          5          0
 deeplabv3plus 0.560251 0.750372   0.277292      0.190121   -0.282959       -1.488308   -0.473080 -1.554257       0.387755   0.183829         16     0         11          0

DELTA_PRED_DICE

## 5. Dimension C — `GLOBAL_PRESERVATION_FP_COST`

Target recovery is always interpreted **together with** the FP / precision cost. Neither component
recovery nor the FP burden is hidden.

In [6]:
C = gate['dimensions']['C']
print('pre-state :', C['pre_state'])
print('state     :', C['state'])
print()
rows = []
for m, c in C['per_model'].items():
    rows.append({'model': m, 'dDice': c['mean_dice_delta'], 'dIoU': c['mean_iou_delta'],
                 'dPrecision': c['mean_precision_delta'], 'dRecall': c['mean_recall_delta'],
                 'meanFPdelta': c['mean_fp_delta'], 'meanFNdelta': c['mean_fn_delta'],
                 'improved': c['cases_improved'], 'unchanged': c['cases_unchanged'],
                 'worsened': c['cases_worsened'], 'maxImp': c['max_case_improvement'],
                 'maxWor': c['max_case_worsening'], 'recovered': c['component_recoveries'],
                 'lost': c['component_losses'], 'empty_rec': c['empty_recoveries']})
print(pd.DataFrame(rows).to_string(index=False))
print()
print(C['evidence'])

pre-state : MIXED_DESCRIPTIVE_EVIDENCE
state     : MATERIAL_GLOBAL_PRESERVATION_FP_COST

         model     dDice      dIoU  dPrecision   dRecall  meanFPdelta  meanFNdelta  improved  unchanged  worsened   maxImp    maxWor  recovered  lost  empty_rec
          unet -0.291866 -0.277550   -0.458950  0.018182  7324.970874   -76.689320        10          0        93 0.386978 -0.789794         16     0          8
attention_unet -0.321186 -0.300327   -0.427001 -0.013473  7528.067961   118.825243         5          0        98 0.388847 -0.756766          9     0          3
        unetpp -0.329161 -0.314980   -0.491628  0.021993  7924.495146  -184.135922         8          0        95 0.381842 -0.799650         14     0          5
 deeplabv3plus -0.282959 -0.264149   -0.494926  0.039483  8334.417476  -403.359223        11          0        92 0.393970 -0.750510         16     0         11

Precision decreases strongly for all four architectures (mean precision delta -0.427001 to -0.494926) whi

## 6. Dimension D — `GENERALITY`

Four frozen architectures; reference model **NONE**; **no winner**; **no ranking**; heterogeneity
reported rather than averaged away.

In [7]:
D = gate['dimensions']['D']
print('pre-state :', D['pre_state'])
print('state     :', D['state'])
print()
for k, v in D['cross_architecture'].items():
    print(f'  {k:<45} {v}')
print()
print(D['evidence'])

pre-state : SUPPORTED_WITH_HETEROGENEITY
state     : CROSS_ARCHITECTURE_CONSISTENT_NEGATIVE__NOT_ISOLATED

  cases_improved_all_four                       3
  cases_nonworse_all_four                       3
  cases_worse_all_four                          86
  components_recovered_by_at_least_one          27
  components_recovered_by_all_four              4
  components_lost_by_at_least_one               0
  components_lost_by_all_four                   0

The negative practical result is present in all four architectures (negative DELTA_PRED_DICE and negative RETENTION_DICE in every architecture; 92-98 of 103 cases worsened per architecture). Cross-architecture: cases worsened for all four = 86, cases improved for all four = 3, cases non-worse for all four = 3; native components recovered by >=1 architecture = 27, recovered by all four = 4, lost by >=1 architecture = 0, lost by all four = 0. The signal is therefore cross-architecture CONSISTENT and is not an isolated architecture-speci

## 7. Prospective logic applied

The frozen outcome rules, applied mechanically. `GO` requires **A and B and C and D**.
`REVISE_LOCALIZER` requires **A supported AND at least one of** (B weak, C materially problematic,
D too architecture-specific). `NO_GO` requires **A no longer scientifically supportable** — and the
frozen invariant forbids `NO_GO` from a poor GT-free localizer result alone while A remains supported.

In [8]:
lg = gate['prospective_logic_applied']
print('GO                 satisfied :', lg['GO']['satisfied'])
print('   blocked by                 :', '; '.join(lg['GO']['blocking']))
print('REVISE_LOCALIZER   satisfied :', lg['REVISE_LOCALIZER']['satisfied'])
print('   triggered by               :', '; '.join(lg['REVISE_LOCALIZER']['triggered_by']))
print('NO_GO              satisfied :', lg['NO_GO']['satisfied'])
print('   blocked by                 :', lg['NO_GO']['blocked_by'])
print()
print('SELECTED VERDICT   :', lg['selected'])

GO                 satisfied : False
   blocked by                 : B weak / fails to translate into deployable utility; C broad systematic degradation undermining the practical benefit
REVISE_LOCALIZER   satisfied : True
   triggered by               : B weak / fails to translate oracle mechanism into deployable utility; C materially problematic global-preservation / FP cost
NO_GO              satisfied : False
   blocked by                 : A remains SUPPORTED_DESCRIPTIVE_EVIDENCE; frozen invariant: a poor GT-free localizer result alone cannot produce NO_GO while A remains supported

SELECTED VERDICT   : REVISE_LOCALIZER


## 8. Final verdict

In [9]:
print('FINAL GATE VERDICT :', gate['final_gate_verdict'])
print()
print('Dimension table (one row per frozen dimension):')
print(dim[['dimension_id', 'dimension_name', 'adjudicated_state',
           'supports_go', 'supports_revise', 'supports_no_go', 'integrity_issue']].to_string(index=False))
print()
print('Reasoning:')
print(gate['verdict_reasoning'])
print()
print('Verdict discipline:')
for k, v in gate['verdict_discipline'].items():
    print(f'  {k} : {v}')

FINAL GATE VERDICT : REVISE_LOCALIZER

Dimension table (one row per frozen dimension):
dimension_id                   dimension_name                                                 adjudicated_state         supports_go     supports_revise supports_no_go integrity_issue
           A           ORACLE_MECHANISM_VALUE                                    SUPPORTED_DESCRIPTIVE_EVIDENCE YES_AS_PRECONDITION YES_AS_PRECONDITION             NO            NONE
           B PRACTICAL_LOCALIZATION_RETENTION WEAK__FAILS_TO_TRANSLATE_ORACLE_MECHANISM_INTO_DEPLOYABLE_UTILITY                  NO                 YES             NO            NONE
           C      GLOBAL_PRESERVATION_FP_COST                              MATERIAL_GLOBAL_PRESERVATION_FP_COST                  NO                 YES             NO            NONE
           D                       GENERALITY              CROSS_ARCHITECTURE_CONSISTENT_NEGATIVE__NOT_ISOLATED                  NO YES_AS_PRECONDITION             NO            NON

## 9. Localization / FP context and claim limitations

The frozen Part-1C proposal context and the Part-1D-B gallery fusion verification are **descriptive
context only**. The gate decision does not require a sole-cause diagnosis.

In [10]:
ctx = gate['localization_fp_context']
print('proposals total                :', ctx['frozen_proposals_total'])
print('overlap >=1 GT pixel           :', ctx['proposals_overlapping_at_least_one_gt_pixel'])
print('overlap no GT pixel            :', ctx['proposals_overlapping_no_gt_pixel'])
print('off-target fraction            :', ctx['off_target_fraction'])
print('gallery pairs checked          :', ctx['gallery_fusion_verification']['case_model_pairs_checked'])
print('outside support P == G exactly :', ctx['gallery_fusion_verification']['outside_support_P_equals_G_exactly'])
print()
print('allowed  :', ctx['allowed_interpretation'])
print('FORBIDDEN:', ctx['forbidden_interpretation'])
print('sole-cause diagnosis required for the gate :', ctx['sole_cause_diagnosis_required_for_gate'])
print()
print('Claim limits:')
for k, v in gate['claim_limits'].items():
    print(f'  {k} : {v}')
print()
print('Phase-13 closure:')
for k, v in gate['phase13_closure'].items():
    print(f'  {k} : {v}')

proposals total                : 494
overlap >=1 GT pixel           : 122
overlap no GT pixel            : 372
off-target fraction            : 0.753036
gallery pairs checked          : 64
outside support P == G exactly : YES

allowed  : descriptive evidence consistent with the current GT-free proposal / integration path driving substantial FP burden
FORBIDDEN: proposal precision is conclusively the sole causal mechanism, or any stronger causal statement unsupported by the experiment
sole-cause diagnosis required for the gate : NO

Claim limits:
  numerical_thresholds_introduced_after_results : 0
  new_subgroup : NO
  weighted_score : NO
  reference_model : None
  ranking : NONE
  winner : NONE
  statistical_superiority_claim : NO
  clinical_superiority_claim : NO
  gallery_used_to_determine_gate : NO
  predeclared_cohorts_only : ['TINY', 'SMALL', 'MEDIUM', 'LARGE', 'hard-Tiny', 'global-empty', 'multi-component']

Phase-13 closure:
  phase13_part1b : COMPLETE_PASS_FROZEN
  phase13_part

## 10. Artifact SHA registry

In [11]:
reg = {
    'gate artifact': AUDITS / 'phase13_regional_integration_gate_after_phase13.json',
    'gate markdown': AUDITS / 'phase13_regional_integration_gate_after_phase13.md',
    'dimension table': TABLES / 'phase13_regional_integration_gate_after_phase13.csv',
    'prospective criteria': AUDITS / 'phase13_regional_integration_gate_criteria_frozen.json',
    'Part-1D R1 protocol': AUDITS / 'phase13_part1d_predicted_region_protocol_frozen_r1.json',
    'Part-1D-B evaluation': AUDITS / 'phase13_part1d_b_predicted_region_evaluation.json',
    'Part-1D-B reproduction': AUDITS / 'phase13_part1d_b_global_reproduction.json',
    'Phase-12 oracle case metrics': TABLES / 'phase12_clean_v3_oracle_case_metrics.csv',
    'Phase-12 oracle component metrics': TABLES / 'phase12_clean_v3_oracle_component_metrics.csv',
    'Phase-12 oracle geometry': TABLES / 'phase12_clean_v3_oracle_geometry.csv',
    'Part-1C proposals': TABLES / 'phase13_part1c_best_checkpoint_proposals.csv',
    'Part-1C case proposals': TABLES / 'phase13_part1c_best_checkpoint_case_proposals.csv',
    'Part-1D-B model summary': TABLES / 'phase13_part1d_predicted_region_model_summary.csv',
    'Part-1D-B case-model': TABLES / 'phase13_part1d_predicted_region_case_model_metrics.csv',
    'Part-1D-B component-model': TABLES / 'phase13_part1d_predicted_region_component_model_metrics.csv',
}
for k, p in reg.items():
    print(f'{sha256_file(p)}  {k}')
print()
print('READ-ONLY: no inference, no model construction, no CUDA, no oracle rerun, no TEST access.')
print('counters:', gate['counters'])

570143a17005b5a2ada232ff4bf8ca6a2980979369d655c2e9f6e96e64af4a8a  gate artifact
fddfdeec5a79b489d00653a498cbb792e28fd2262f74424b779858cf7262834d  gate markdown
f0d0d8d6b89cf274f074d9d94f64576dc4a487dc571f8cab803e9edcde60a5ba  dimension table
342a8bcac2c87bf0b85f7bbd963179e74f029825c0434cf527fbd939d2d13c21  prospective criteria
9089b948ef7563252963bf68d0e0b57a9e891c5e913f54a315bce94c98207eb8  Part-1D R1 protocol
9a9838ac64af5109c13e0625066e6b891068c93ef5b726f89f03a65e4828ad28  Part-1D-B evaluation
a2914fb7c67cb68c8b60074fbf52fc1b424707201af65e60aee7c36a8680b82f  Part-1D-B reproduction
080f158fb8b3e3552ac7854c6d9a116e11564bcefa9efdcdbb3511dba090b436  Phase-12 oracle case metrics
d66933a3c2821d3ab9375eadb5c0540fea145edcf49b648064fe9aa4c460c753  Phase-12 oracle component metrics
29326d5b9a3bf71add74d8743750511f146d8c4dfbd1dc5b04a5e1b647c37cc0  Phase-12 oracle geometry
8c3151a3f1f9a56f0692c6ff2d2171472a777a56e99fa2e474aff8a3478a564a  Part-1C proposals
d25c44ec1361365e261d36458e86e707ad3bcb5